# Tutorial: Build a Shapash Regression Report

This notebook walks through the full construction of a **Shapash regression report** step by step.

We will start from a simple `RandomForestRegressor`, compile a `SmartExplainer`, generate a base report from the default regression YAML template, then extend the report with custom blocks.

The tutorial is designed for users discovering the product, so each step includes comments explaining why it is needed and how to adapt it to your own project.

At the end of the notebook, you will have:
- a trained regression model
- a compiled `SmartExplainer`
- a base HTML report generated from `default_report_regression_house_prices.yml`
- a custom HTML report generated from `custom_report_regression_house_prices.yml` with user-defined blocks

## 1. Imports and Working Directories

We first import the libraries needed for data preparation, model training, explainability, and report generation.

The small directory helper below makes the notebook robust whether it is executed from the repository root or directly from the `tutorial/generate_report` folder.

In [ ]:
from pathlib import Path

import pandas as pd
from category_encoders import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from shapash import SmartExplainer
from shapash.data.data_loader import data_loading
from shapash.report import ReportTemplate, export_report_yml
from shapash.report.blocks import ReportBlockMixin, block

BASE_DIR = Path.cwd()
if not (BASE_DIR / "config").exists():
    BASE_DIR = BASE_DIR / "tutorial" / "generate_report"

CONFIG_DIR = BASE_DIR / "config"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base directory: {BASE_DIR}")
print(f"Config directory: {CONFIG_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## 2. Load the Dataset

We use the built-in **House Prices** dataset shipped with Shapash tutorials.

The target is `SalePrice`. All other columns are used as input features.

In [ ]:
house_df, house_dict = data_loading("house_prices")

y_df = house_df["SalePrice"]
x_df = house_df[house_df.columns.difference(["SalePrice"])].copy()

print(f"Dataset shape: {house_df.shape}")
print(f"Number of features: {x_df.shape[1]}")
house_df.head()

## 3. Prepare the Data and Train a Simple Regression Model

To keep the example simple, we use an `OrdinalEncoder` for categorical variables and then train a `RandomForestRegressor`.

This is intentionally a lightweight baseline model: the focus of this notebook is the **report generation workflow**, not model optimization.

In [ ]:
# Convert non-numeric columns to object so the encoder treats them as categorical features.
for col in x_df.columns:
    if not pd.api.types.is_numeric_dtype(x_df[col]):
        x_df[col] = x_df[col].astype(object)

categorical_features = [
    col
    for col in x_df.columns
    if pd.api.types.is_object_dtype(x_df[col]) or pd.api.types.is_string_dtype(x_df[col])
]

encoder = OrdinalEncoder(
    cols=categorical_features,
    handle_unknown="return_nan",
    return_df=True,
).fit(x_df)

x_df_encoded = encoder.transform(x_df)

xtrain, xtest, ytrain, ytest = train_test_split(
    x_df_encoded,
    y_df,
    train_size=0.75,
    random_state=1,
)

regressor = RandomForestRegressor(
    n_estimators=200,
    random_state=1,
).fit(xtrain, ytrain)

# Keep predictions as a DataFrame to match report expectations.
y_pred = pd.DataFrame(regressor.predict(xtest), columns=["pred"], index=xtest.index)

print(f"Train shape: {xtrain.shape}")
print(f"Test shape: {xtest.shape}")
print(f"MAE: {mean_absolute_error(ytest, y_pred.iloc[:, 0]):,.2f}")
print(f"MSE: {mean_squared_error(ytest, y_pred.iloc[:, 0]):,.2f}")
print(f"R2: {r2_score(ytest, y_pred.iloc[:, 0]):.3f}")

## 4. Instantiate and Compile the SmartExplainer

`SmartExplainer` is the central Shapash object used to compute explainability artifacts and later generate the report.

Compilation links the model outputs, the dataset to explain, and the human-readable feature labels.

In [ ]:
xpl = SmartExplainer(
    model=regressor,
    preprocessing=encoder,
    features_dict=house_dict,
)

# Compile once before generating any report.
xpl.compile(x=xtest, y_pred=y_pred, y_target=ytest)

xpl

## 5. Generate a Base Regression Report

We now generate a first report using the default regression template.

This gives a complete report using **built-in blocks only**.

In [ ]:
base_report_path = OUTPUT_DIR / "default_regression_report.html"

xpl.generate_report(
    output_file=str(base_report_path),
    x_train=xtrain,
    y_train=ytrain,
    y_test=ytest,
)

print(f"Base report generated: {base_report_path}")

## 6. Inspect and Adapt the YAML Configuration

Shapash ships with default regression and classification report templates.

Using `export_report_yml`, you can copy a template into your own working directory and then adapt it to your project. This is the recommended starting point for most users.

Before generating the report, open the exported YAML file and adapt it to your own context.

Typical customizations include:
- changing the report title and subtitle
- updating project information and dataset information
- documenting your data preparation choices
- editing the list of blocks and their parameters
- choosing the metrics to display

The next cell prints the template content so you can inspect it directly from the notebook.

In [ ]:
default_template_path = CONFIG_DIR / "default_report_regression_tutorial.yml"

# Export the built-in default regression template to a local file.
export_report_yml(ReportTemplate.DEFAULT_REGRESSION, output_path=str(default_template_path))

print(f"Default template exported to: {default_template_path}")

# Read the YAML template as plain text so users can review it before editing.
print(default_template_path.read_text(encoding="utf-8"))

## 7. Create a Class with Custom Report Blocks

One of the strengths of the Shapash reporting system is that you can extend it with your own blocks.

A custom block is simply a method named `block_<type>` inside a class inheriting from `ReportBlockMixin`.

These blocks can use the explainer data, the predictions, the targets, and any additional logic you want to expose in the report.

Decorated methods can return either ``(title, body)`` or a bare body value.

The body may be a single supported item or a list of supported items.
Each item can be a string, a pandas ``DataFrame``, a Plotly figure, or a Panel viewable.
Tuples inside the body are rendered as horizontal rows.

In [ ]:
class CustomRegressionReportBlocks(ReportBlockMixin):
    """User-defined blocks for a house prices regression explainability report."""

    @block
    def block_residual_error_summary(self, title: str = "Residual error summary"):
        """Summarize residual dispersion and global regression performance indicators."""
        if self.y_test is None or self.y_pred is None:
            raise ValueError("residual_error_summary block requires y_test and y_pred.")

        # Convert arrays to aligned pandas Series for simple metrics computation.
        y_true = pd.Series(self.y_test).reset_index(drop=True)
        y_pred_series = pd.Series(self.y_pred).reset_index(drop=True)
        residuals = y_true - y_pred_series
        abs_residuals = residuals.abs()

        summary_df = pd.DataFrame(
            [
                ["MAE", f"{mean_absolute_error(y_true, y_pred_series):,.2f}"],
                ["MSE", f"{mean_squared_error(y_true, y_pred_series):,.2f}"],
                ["R2", f"{r2_score(y_true, y_pred_series):.3f}"],
                ["Residual mean", f"{residuals.mean():,.2f}"],
                ["Residual std", f"{residuals.std():,.2f}"],
                ["Median absolute error", f"{abs_residuals.median():,.2f}"],
                ["95th pct absolute error", f"{abs_residuals.quantile(0.95):,.2f}"],
            ],
            columns=["Metric", "Value"],
        )

        explanation = (
            "This section summarizes global regression error levels. "
            "A strong gap between median and 95th percentile absolute error usually highlights "
            "a subset of difficult cases worth deeper explainability analysis."
        )
        return title, [explanation, summary_df]

    @block
    def block_largest_errors_focus(self, title: str = "Largest absolute errors", top_k: int = 10):
        """Display samples with the largest absolute errors to prioritize local explainability reviews."""
        explainer = self._require_explainer("largest_errors_focus")
        if self.y_test is None or self.y_pred is None:
            raise ValueError("largest_errors_focus block requires y_test and y_pred.")

        # Rebuild a detailed table mixing predictions, residuals, and business context columns.
        y_true = pd.Series(self.y_test, index=explainer.x_init.index, name="true")
        y_pred_series = pd.Series(self.y_pred, index=explainer.x_init.index, name="pred")
        details = pd.concat([y_true, y_pred_series, explainer.x_init], axis=1)
        details["residual"] = details["true"] - details["pred"]
        details["abs_error"] = details["residual"].abs()

        focus = details.sort_values("abs_error", ascending=False).head(top_k).copy()
        if focus.empty:
            return title, ["No rows available to compute largest errors."]

        focus = focus.rename(
            columns={"true": "True", "pred": "Pred", "residual": "Residual", "abs_error": "AbsError"}
        )

        # Keep a few business-relevant columns first when they are available.
        preferred = ["OverallQual", "GrLivArea", "TotalBsmtSF", "GarageArea", "Neighborhood"]
        context_cols = [c for c in preferred if c in focus.columns]
        leading = ["True", "Pred", "Residual", "AbsError"]
        trailing = [c for c in focus.columns if c not in leading + context_cols]
        focus = focus[leading + context_cols + trailing]

        for col in ["True", "Pred", "Residual", "AbsError"]:
            focus[col] = focus[col].map(lambda x: round(float(x), 2))

        info = (
            "Rows below are the largest absolute errors. "
            "They are ideal candidates for local contribution plots and feature-level investigation."
        )
        table = focus.reset_index(drop=False)
        return title, [info, table]

## 8. Generate a Custom Regression Report

We now use the custom YAML layout already written for this tutorial and the custom block class defined above.

This second report combines built-in Shapash sections with project-specific regression diagnostics.

In [ ]:
custom_report_path = OUTPUT_DIR / "regression_report_custom.html"
custom_config_path = CONFIG_DIR / "custom_report_regression_house_prices.yml"

custom_blocks = CustomRegressionReportBlocks(
    explainer=xpl,
    x_train=xtrain,
    y_train=ytrain,
    y_test=ytest,
)

xpl.generate_report(
    output_file=str(custom_report_path),
    yaml_path=str(custom_config_path),
    block_instance=custom_blocks,
)

print(f"Custom report generated: {custom_report_path}")

## 9. What to Customize Next

Once this notebook works end to end, the usual next step is to adapt it to your own project.

In practice, you will typically modify:
- the dataset loading logic
- the preprocessing pipeline
- the trained model
- the default or custom YAML configuration
- the custom report blocks needed by your use case

Open the generated HTML files in your browser to review the result and iterate on the YAML and custom blocks.